# CNN-Based Image Classification — Super Mario Bros Level Classifier

This notebook explores the design, training, and evaluation of two Convolutional Neural Network (CNN) architectures for binary image classification. The task is to distinguish between screenshots from **Level 1** and **Level 8** of *Super Mario Bros* using supervised deep learning.

## Objectives
- Build and train two CNN architectures of different complexity on a labeled image dataset.
- Compare their classification performance, training time, and parameter count.
- Evaluate each model using accuracy, binary cross-entropy, and confusion matrices.

## Dataset
The dataset (`SuperMarioBros_Dataset`) consists of labeled screenshots organized into training, validation, and test splits, with two classes: `Lvl1` and `Lvl8`. Images are resized to **224×224 pixels** and normalized before being fed into the models.

## Dataset Directory Structure
The dataset should be organized as follows before running this notebook:

```
SuperMarioBros_Dataset/
├── TrainImages/
│   ├── TrainLvl1/
│   └── TrainLvl8/
├── ValidationImages/
│   ├── ValidationLvl1/
│   └── ValidationLvl8/
└── TestImages/
    ├── TestLvl1/
    └── TestLvl8/
```


## 1. Imports

All required libraries are imported below. TensorFlow/Keras is used for model building and training; OpenCV and NumPy handle image preprocessing; scikit-learn provides evaluation metrics.

In [ ]:
from __future__ import print_function

import os
import itertools

import cv2
import numpy as np
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.utils import to_categorical
from keras.layers import Dense, Conv2D, MaxPooling2D, Dropout, Flatten
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
import keras.backend as K


## 2. Helper Functions

The function below renders a confusion matrix as a heatmap using Matplotlib. It supports optional normalization and annotates each cell with its count.

> **Source:** Adapted from [DeepLizard](https://deeplizard.com/learn/video/km7pxKy4UHU)


In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion Matrix',
                          cmap=plt.cm.Blues):
    """
    Plot a confusion matrix with class labels and color coding.

    Args:
        cm (ndarray): Confusion matrix from sklearn.
        classes (list): List of class label names.
        normalize (bool): If True, normalize counts to proportions.
        title (str): Title displayed above the plot.
        cmap: Matplotlib colormap.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment='center',
                 color='white' if cm[i, j] > thresh else 'black')

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')


## 3. Data Loading

The dataset is loaded from the local `data/` directory. See the [README](../README.md) for the expected folder structure.


In [ ]:
import os

# Paths are relative to the notebook location (notebooks/)
DATA_ROOT       = os.path.join('..', 'data')
TRAIN_PATH      = os.path.join(DATA_ROOT, 'TrainImages')
TEST_PATH       = os.path.join(DATA_ROOT, 'TestImages')
VALIDATION_PATH = os.path.join(DATA_ROOT, 'ValidationImages')

# Quick sanity check
for name, path in [('Train', TRAIN_PATH), ('Test', TEST_PATH), ('Validation', VALIDATION_PATH)]:
    status = '✓ found' if os.path.exists(path) else '✗ NOT FOUND — check data/ folder'
    print(f'{name:12}: {path}  [{status}]')


## 4. Image Preprocessing

Images are loaded using Keras' `ImageDataGenerator`, which streams batches from disk — avoiding memory overflow when working with large image datasets. All pixel values are rescaled from `[0, 255]` to `[0, 1]` by dividing by 255.

The training and validation generators use `shuffle=True` to randomize the sample order during training, while the test generator uses `shuffle=False` to ensure prediction order matches the ground-truth labels.


In [ ]:
IMAGE_SIZE = (224, 224)

# Initialize generators with pixel normalization
train_datagen      = ImageDataGenerator(rescale=1.0 / 255)
test_datagen       = ImageDataGenerator(rescale=1.0 / 255)
validation_datagen = ImageDataGenerator(rescale=1.0 / 255)

# Create data generators that stream images from directory
train_gen = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=IMAGE_SIZE, class_mode='categorical', shuffle=True
)
test_gen = test_datagen.flow_from_directory(
    TEST_PATH, target_size=IMAGE_SIZE, class_mode='categorical', shuffle=False
)
validation_gen = validation_datagen.flow_from_directory(
    VALIDATION_PATH, target_size=IMAGE_SIZE, class_mode='categorical', shuffle=True
)


---
## 5. Model 1 — Deep CNN (High Complexity)

### 5.1 Architecture

The first model follows a classic deep CNN pattern with three convolutional blocks of increasing filter depth, followed by two large fully connected layers.

| Block | Layer Type | Filters / Units | Kernel | Activation |
|-------|-----------|-----------------|--------|------------|
| 1 | Conv2D × 2 | 32 | 3×3 | ReLU |
| 1 | MaxPooling2D | — | 2×2 | — |
| 1 | Dropout | — | — | p=0.25 |
| 2 | Conv2D × 2 | 64 | 3×3 | ReLU |
| 2 | MaxPooling2D | — | 2×2 | — |
| 2 | Dropout | — | — | p=0.25 |
| 3 | Conv2D × 2 | 128 | 3×3 | ReLU |
| 3 | MaxPooling2D | — | 2×2 | — |
| 3 | Dropout | — | — | p=0.25 |
| FC | Dense × 2 | 1000 | — | ReLU |
| FC | Dropout | — | — | p=0.50 |
| Out | Dense | 2 | — | Sigmoid |

**Design decisions:**
- **MaxPooling** reduces spatial dimensions while retaining dominant features.
- **Dropout** at 25% after each pooling block mitigates overfitting.
- **Sigmoid** output is appropriate for binary classification, mapping predictions to `[0, 1]`.


In [ ]:
# Input configuration
N_ROWS, N_COLS, N_DIMS = 224, 224, 3
INPUT_SHAPE = (N_ROWS, N_COLS, N_DIMS)
N_CLASSES = 2  # Lvl1 and Lvl8

def build_model_1():
    """
    Construct the first CNN architecture — three conv blocks with
    32/64/128 filters, followed by two dense layers of 1000 units each.
    """
    model = Sequential()

    # Block 1 — 32 filters
    model.add(Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=INPUT_SHAPE))
    model.add(Conv2D(32, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 2 — 64 filters
    model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 3 — 128 filters
    model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Fully connected head
    model.add(Flatten())
    model.add(Dense(1000, activation='relu'))
    model.add(Dense(1000, activation='relu'))
    model.add(Dropout(0.5))

    # Output layer
    model.add(Dense(N_CLASSES, activation='sigmoid'))

    return model


### 5.2 Compilation

The model is compiled with the **Adam** optimizer — a strong default for image tasks with large datasets — and **binary cross-entropy** as the loss function, which is standard for binary classification.


In [ ]:
BATCH_SIZE = 256
EPOCHS     = 5

model1 = build_model_1()
model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'binary_crossentropy']
)
model1.summary()


> **Note:** This architecture has over 80 million trainable parameters — the bulk of which reside in the first dense layer. The large fully connected head is the primary driver of model size.

### 5.3 Training

The model is trained for 5 epochs using the training generator, with the validation generator used to monitor generalization after each epoch.

In [ ]:
history1 = model1.fit_generator(
    generator=train_gen,
    epochs=EPOCHS,
    verbose=1,
    validation_data=validation_gen
)


**Training results (epoch 5):**
- Train accuracy: **99.49%** — Binary cross-entropy: **0.0261**
- Validation accuracy: **97.53%** — Binary cross-entropy: **0.7981**


### 5.4 Evaluation

The trained model is evaluated on the held-out test set.

In [ ]:
model1.evaluate(test_gen)

The model achieves **100% accuracy** on the test set — every image is classified correctly. The confusion matrix below confirms this.

In [ ]:
# Extract class labels
target_names = list(test_gen.class_indices.keys())

# Generate predictions
Y_pred1 = model1.predict_generator(test_gen)
y_pred1  = np.argmax(Y_pred1, axis=1)

# Compute and plot confusion matrix
cm1 = confusion_matrix(test_gen.classes, y_pred1)
plot_confusion_matrix(cm1, target_names, title='Model 1 — Confusion Matrix')


All 1,500 test images fall along the diagonal — confirming zero misclassifications.

### 5.5 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1.history['accuracy'], marker='o')
axes[0].set_title('Model 1 — Training Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True)

axes[1].plot(history1.history['binary_crossentropy'], marker='o', color='tomato')
axes[1].set_title('Model 1 — Binary Cross-Entropy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True)

plt.tight_layout()
plt.show()


Both metrics converge rapidly. Accuracy is near 100% from the first epoch, and cross-entropy drops well below 0.2 — a threshold generally considered strong performance. The model learns the underlying patterns quickly and cleanly.

### Summary — Model 1
Model 1 achieves perfect test classification with a 100% accuracy rate. While this is an exceptional result, the ~80M parameter count and ~22-minute training time suggest the architecture may be over-engineered for this dataset. This motivates exploring a lighter alternative.


---
## 6. Model 2 — Lighter CNN (Reduced Complexity)

### 6.1 Architecture

Model 2 explores whether similar classification performance can be achieved with a significantly smaller architecture. The key changes from Model 1 are:

- **Fewer filters per conv block:** 20 / 50 / 90 instead of 32 / 64 / 128.
- **Deeper but narrower dense head:** 5 layers (500 → 300 → 300 → 200 → 100 units) instead of 2 layers of 1000 units each.
- **Lower dropout rate:** 10% throughout (vs. 25%/50% in Model 1), reflecting a reduced risk of overfitting in a lighter model.

| Block | Layer Type | Filters / Units | Kernel | Activation |
|-------|-----------|-----------------|--------|------------|
| 1 | Conv2D × 2 | 20 | 3×3 | ReLU |
| 1 | MaxPooling2D | — | 2×2 | — |
| 1 | Dropout | — | — | p=0.10 |
| 2 | Conv2D × 2 | 50 | 3×3 | ReLU |
| 2 | MaxPooling2D | — | 2×2 | — |
| 2 | Dropout | — | — | p=0.10 |
| 3 | Conv2D × 2 | 90 | 3×3 | ReLU |
| 3 | MaxPooling2D | — | 2×2 | — |
| 3 | Dropout | — | — | p=0.10 |
| FC | Dense (500 → 300 → 300 → 200 → 100) | — | — | ReLU |
| FC | Dropout | — | — | p=0.10 |
| Out | Dense | 2 | — | Sigmoid |


In [ ]:
def build_model_2():
    """
    Construct the second CNN architecture — three conv blocks with
    20/50/90 filters, followed by a deeper but narrower dense head.
    """
    model = Sequential()

    # Block 1 — 20 filters
    model.add(Conv2D(20, (3, 3), padding='same', activation='relu', input_shape=INPUT_SHAPE))
    model.add(Conv2D(20, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.1))

    # Block 2 — 50 filters
    model.add(Conv2D(50, (3, 3), padding='same', activation='relu'))
    model.add(Conv2D(50, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.1))

    # Block 3 — 90 filters
    model.add(Conv2D(90, (3, 3), padding='same', activation='relu'))
    model.add(Conv2D(90, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.1))

    # Fully connected head — deeper, narrower
    model.add(Flatten())
    model.add(Dense(500, activation='relu'))
    model.add(Dense(300, activation='relu'))
    model.add(Dense(300, activation='relu'))
    model.add(Dense(200, activation='relu'))
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(0.1))

    # Output layer
    model.add(Dense(N_CLASSES, activation='sigmoid'))

    return model


### 6.2 Compilation

In [ ]:
model2 = build_model_2()
model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'binary_crossentropy']
)
model2.summary()


> **Note:** Model 2 has approximately 31M trainable parameters — about **one-third** of Model 1's parameter count.

### 6.3 Training

In [ ]:
history2 = model2.fit_generator(
    generator=train_gen,
    epochs=EPOCHS,
    verbose=1,
    validation_data=validation_gen
)


**Training results (epoch 5):**
- Train accuracy: **99.85%** — Binary cross-entropy: **0.0042**
- Validation accuracy: **96.53%** — Binary cross-entropy: **1.4377**

Model 2 trains in approximately **3 minutes** versus ~22 minutes for Model 1 — a **7× speedup** for a model with one-third the parameters.


### 6.4 Evaluation

In [ ]:
model2.evaluate(test_gen)

Model 2 achieves **99.87% accuracy** on the test set — only 2 images out of 1,500 are misclassified. The confusion matrix below provides the full breakdown.

In [ ]:
target_names = list(test_gen.class_indices.keys())

Y_pred2 = model2.predict_generator(test_gen)
y_pred2  = np.argmax(Y_pred2, axis=1)

cm2 = confusion_matrix(test_gen.classes, y_pred2)
plot_confusion_matrix(cm2, target_names, title='Model 2 — Confusion Matrix')


Both Level 8 images are classified correctly. The 2 misclassified images are Level 1 screenshots predicted as Level 8 — an extremely low error rate.

### 6.5 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history2.history['accuracy'], marker='o')
axes[0].set_title('Model 2 — Training Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True)

axes[1].plot(history2.history['binary_crossentropy'], marker='o', color='tomato')
axes[1].set_title('Model 2 — Binary Cross-Entropy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True)

plt.tight_layout()
plt.show()


Model 2 converges slightly slower in early epochs compared to Model 1, but catches up by epoch 3. By epoch 5, cross-entropy values are actually lower than Model 1's — suggesting better fine-grained learning despite fewer parameters.

### Summary — Model 2
Despite having one-third the parameters and training in one-tenth the time, Model 2 achieves near-perfect test accuracy (99.87%). The trade-off of a slightly wider but shallower convolutional backbone, paired with a deeper and narrower dense head and lower dropout, proves highly effective on this dataset.


---
## 7. Model Comparison

| Metric | Model 1 | Model 2 |
|--------|---------|---------|
| Conv filters | 32 / 64 / 128 | 20 / 50 / 90 |
| Dense layers | 2 × 1000 | 500 → 300 → 300 → 200 → 100 |
| Dropout rate | 0.25 / 0.50 | 0.10 |
| Parameters | ~80M | ~31M |
| Training time (5 epochs) | ~22 min | ~3 min |
| Test accuracy | **100%** | **99.87%** |
| Test cross-entropy | lower | slightly higher |
| Misclassified (test) | 0 | 2 |

Both models perform exceptionally well on this binary classification task. Model 1 achieves a perfect score, while Model 2 closely matches it at a fraction of the computational cost.


---
## 8. Conclusions

1. **Model 1** achieved perfect test classification (100% accuracy), demonstrating that a deep CNN with high filter counts and large fully connected layers is capable of learning the visual differences between Mario levels without overfitting.

2. **Model 2** achieved 99.87% test accuracy with one-third the parameters and one-tenth the training time. This confirms that the first architecture, while excellent, is over-parameterized for this task.

3. **Architectural trade-offs matter:** Reducing convolutional filter counts required compensating with a deeper dense head and lower dropout rates to maintain performance. This suggests that when reducing model width, increasing depth in the classifier can partially recover representational power.

4. **Dataset complexity:** Both models trained to near-perfect performance in just 5 epochs on a relatively small training set. This indicates the dataset is visually predictable — the pixel-level differences between Level 1 and Level 8 are likely strong enough that even lightweight models generalize well. Results on a more diverse or ambiguous dataset would differ.

5. **Practical recommendation:** Model 2 is the preferred choice for deployment. The marginal accuracy difference (0.13%) does not justify Model 1's 7× longer training time and 2.5× larger parameter footprint. Model 2 is faster to train, smaller to store, and easier to iterate on.

6. **Further work:** Testing with a richer or noisier dataset, applying data augmentation, or extending to multi-class level classification would better stress-test both architectures and reveal where their performance diverges.
